**Corrective RAG Implementation**

Some Important Specifications -
This notebook implements a basic Naive RAG (Retrieval-Augmented Generation) pipeline using:

1. Supabase as vector database
2. PyMuPDF for PDF reading
3. Sentence Transformers embeddings
4. BAAI/bge-small-en-v1.5 embedding model
5. vecs as pgvector wrapper
6. Groq for LLM inference

This colab notebook is a breif introduction to how a normal RAG model works for starters. A production-grade RAG is different in some aspects while the logic remains the same.

#Overview

While Self RAG evaluates a response after the answer is generated based on retrival decision,  document relevance, groundness check and completeness check, A corrective RAG actually evaluates the documents that are retrived before the response is generated, it can be in the form of -
1. Are retrieved docs reliable?
2. Are they relevant enough?
3. Is retrieval confidence low?


Now this document relevance check is done based on three methods -
Retrival Evaluation

1.   Similarity Score Thresholding
2.   List item


Similarity Score Thresholding
Cross Encoder Re-Ranking - Sentence Transformers
LLM as a Judge Retrieval Evaluation - LLM Prompting
Query Coverage Evaluation - LLM Prompting
Context Richment
Query ReWritting



HyDE
Multi-Query Retrieval
Query Decomposition
Hybrid Retrieval (*****DOUBT******)
Web Search Fallback (Search APIs (most common), Search engine wrappers/libraries, Self-hosted search infrastructure, Browser agents / scraping fallback)
Context Richment
Chunk Filtering
Compression
Metadata Grounding
Citation Alignment
| Method                                        | Complexity.   | Common?                 |
| -----------------------------      | ----------      | ----------------        |
| Source-aware prompting            | Easy              | VERY common         |
| Embedding similarity matching  | Medium         | Common                  |
| Token/span attribution                | Hard              | Advanced systems  |

**Model Workflow**

User Query -> Query Rewrite -> HyDE + Multi-query expansion -> Hybrid Retrieval -> Cross-Encoder Rerank -> Retrieval Evaluation -> Fallback Web Search -> Chunk Filtering + Compression -> Grounded Generation -> Citation Alignment -> Groundedness Verification -> Final Answer

Below is a realistic production-style CRAG pipeline in Python.
It includes:
1. Query rewriting
2. HyDE
3. Multi-query retrieval
4. Hybrid retrieval
5. Cross-encoder reranking
6. Retrieval evaluation
7. Web fallback
8. Chunk filtering
9. Context compression
10. Citation alignment
11. Grounded answer generation
12. Verification

it is close example of a production grade RAG.

#Step 1: Install Libraries

In [ ]:
pip install \
langchain \
langchain-openai \
sentence-transformers \
qdrant-client \
rank-bm25 \
numpy \
scikit-learn \
tavily-python \
nltk

In [ ]:
import uuid
import numpy as np
import nltk

from typing import List, Dict
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

from qdrant_client import QdrantClient
from qdrant_client.models import Filter

from tavily import TavilyClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

#Step 2: Supabase, LLM, Embedding Model SetUp

In [ ]:
OPENAI_API_KEY = "YOUR_KEY"
TAVILY_API_KEY = "YOUR_KEY"

VECTOR_COLLECTION = "documents"

TOP_K = 20
FINAL_TOP_K = 5

SIMILARITY_THRESHOLD = 0.72
RERANK_THRESHOLD = 0.30

In [ ]:

llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0
)

embeddings = OpenAIEmbeddings()

reranker = CrossEncoder(
    "BAAI/bge-reranker-large"
)

qdrant = QdrantClient(
    host="localhost",
    port=6333
)

tavily = TavilyClient(
    api_key=TAVILY_API_KEY
)

#Step 3: Query Re-Write

In [ ]:
def rewrite_query(query: str) -> str:

    prompt = f"""
    Rewrite the query for optimal retrieval.

    Expand acronyms.
    Clarify intent.
    Add missing technical specificity.

    Query:
    {query}
    """

    return llm.invoke(prompt).content.strip()

#Step 4: HyDE + Multi-Query Generation

In [ ]:
def generate_hyde(query: str) -> str:

    prompt = f"""
    Write a detailed ideal answer for:
    {query}
    """

    return llm.invoke(prompt).content.strip()

In [ ]:
def generate_multi_queries(query: str) -> List[str]:

    prompt = f"""
    Generate 5 semantic search variations for:

    {query}
    """

    response = llm.invoke(prompt).content

    return [
        q.strip("- ").strip()
        for q in response.split("\n")
        if q.strip()
    ]

#Step 5: Hybrid Retrieval

In [ ]:

def vector_search(query: str, top_k=TOP_K):

    query_embedding = embeddings.embed_query(query)

    results = qdrant.search(
        collection_name=VECTOR_COLLECTION,
        query_vector=query_embedding,
        limit=top_k
    )

    docs = []

    for r in results:

        docs.append({
            "id": r.id,
            "text": r.payload["text"],
            "metadata": r.payload["metadata"],
            "score": r.score
        })

    return docs

In [ ]:

def bm25_search(query: str, corpus):

    tokenized_corpus = [
        doc["text"].split()
        for doc in corpus
    ]

    bm25 = BM25Okapi(tokenized_corpus)

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    ranked = sorted(
        zip(corpus, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [x[0] for x in ranked[:TOP_K]]

In [ ]:
def hybrid_retrieve(query, corpus):

    vector_docs = vector_search(query)

    bm25_docs = bm25_search(query, corpus)

    merged = {}

    for d in vector_docs + bm25_docs:
        merged[d["id"]] = d

    return list(merged.values())

#Step 6: Cross-Encoder ReRank

In [ ]:
def rerank(query, docs):

    pairs = [
        (query, d["text"])
        for d in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    filtered = []

    for doc, score in ranked:

        if score >= RERANK_THRESHOLD:

            doc["rerank_score"] = float(score)

            filtered.append(doc)

    return filtered[:FINAL_TOP_K]

#Step 7: Retrival Evaluation

In [ ]:
def evaluate_retrieval(docs):

    if not docs:
        return False

    avg_score = np.mean([
        d["rerank_score"]
        for d in docs
    ])

    return avg_score >= SIMILARITY_THRESHOLD

#Step 8: Web-Search Fallback

In [ ]:
def web_search(query):

    results = tavily.search(
        query=query,
        max_results=5
    )

    docs = []

    for r in results["results"]:

        docs.append({
            "id": str(uuid.uuid4()),
            "text": r["content"],
            "metadata": {
                "source": r["url"],
                "title": r["title"]
            },
            "score": r["score"]
        })

    return docs

#Step 9: Chunking Filtering

In [ ]:
def filter_chunks(docs):

    seen = set()

    filtered = []

    for d in docs:

        text_hash = hash(d["text"])

        if text_hash not in seen:

            filtered.append(d)

            seen.add(text_hash)

    return filtered


#Step 10: Context Compression

In [ ]:

def compress_chunks(query, docs):

    compressed = []

    for d in docs:

        prompt = f"""
        Extract ONLY information relevant to:

        {query}

        TEXT:
        {d["text"]}
        """

        summary = llm.invoke(prompt).content

        d["compressed_text"] = summary

        compressed.append(d)

    return compressed


#Step 11: Context Assembly

In [ ]:
def build_context(docs):

    blocks = []

    for i, d in enumerate(docs):

        source_id = f"SOURCE_{i}"

        d["source_id"] = source_id

        block = f"""
        [{source_id}]
        {d['compressed_text']}
        """

        blocks.append(block)

    return "\n\n".join(blocks)

#Step 12: Grounded Generation

In [ ]:
def generate_answer(query, context):

    prompt = f"""
    You are a grounded AI assistant.

    ONLY answer using provided context.

    Every factual claim MUST include citations.

    Context:
    {context}

    Query:
    {query}
    """

    return llm.invoke(prompt).content

#Step 13: Citation Alignment

In [ ]:
def align_citations(answer, docs):

    sentences = nltk.sent_tokenize(answer)

    aligned = []

    chunk_embeddings = []

    for d in docs:

        emb = embeddings.embed_query(
            d["compressed_text"]
        )

        chunk_embeddings.append(
            (d, emb)
        )

    for sentence in sentences:

        sent_emb = embeddings.embed_query(sentence)

        best_doc = None
        best_score = -1

        for doc, chunk_emb in chunk_embeddings:

            score = cosine_similarity(
                [sent_emb],
                [chunk_emb]
            )[0][0]

            if score > best_score:

                best_score = score
                best_doc = doc

        aligned.append(
            f"{sentence} [{best_doc['source_id']}]"
        )


    return "\n".join(aligned)

#Step 14: Answer Verification

In [ ]:
def verify_grounding(answer, context):

    prompt = f"""
    Determine whether the answer is fully
    supported by the provided context.

    Context:
    {context}

    Answer:
    {answer}

    Output ONLY:
    YES or NO
    """

    result = llm.invoke(prompt).content.strip()

    return result == "YES"

#Step 15: Main C-RAG Pipeline

In [ ]:
def production_crag(query, corpus):

    # -------------------------
    # Rewrite
    # -------------------------

    rewritten = rewrite_query(query)

    # -------------------------
    # HyDE
    # -------------------------

    hyde_doc = generate_hyde(rewritten)

    # -------------------------
    # Multi-query expansion
    # -------------------------

    multi_queries = generate_multi_queries(
        rewritten
    )

    multi_queries.append(hyde_doc)

    # -------------------------
    # Hybrid retrieval
    # -------------------------

    retrieved = []

    for q in multi_queries:

        docs = hybrid_retrieve(q, corpus)

        retrieved.extend(docs)

    # -------------------------
    # Deduplicate
    # -------------------------

    unique = {}

    for d in retrieved:
        unique[d["id"]] = d

    retrieved = list(unique.values())

    # -------------------------
    # Rerank
    # -------------------------

    reranked = rerank(query, retrieved)

    # -------------------------
    # Evaluate retrieval quality
    # -------------------------

    retrieval_ok = evaluate_retrieval(
        reranked
    )

    # -------------------------
    # Web fallback
    # -------------------------

    if not retrieval_ok:

        web_docs = web_search(query)

        reranked.extend(web_docs)

        reranked = rerank(query, reranked)

    # -------------------------
    # Filter chunks
    # -------------------------

    filtered = filter_chunks(reranked)

    # -------------------------
    # Compress
    # -------------------------

    compressed = compress_chunks(
        query,
        filtered
    )

    # -------------------------
    # Build context
    # -------------------------

    context = build_context(compressed)

    # -------------------------
    # Generate answer
    # -------------------------

    answer = generate_answer(
        query,
        context
    )

    # -------------------------
    # Citation alignment
    # -------------------------

    aligned = align_citations(
        answer,
        compressed
    )

    # -------------------------
    # Grounding verification
    # -------------------------

    verified = verify_grounding(
        aligned,
        context
    )

    if not verified:

        return {
            "success": False,
            "message": "Grounding verification failed."
        }

    return {
        "success": True,
        "query": query,
        "rewritten_query": rewritten,
        "answer": aligned,
        "sources": [
            {
                "source_id": d["source_id"],
                "metadata": d["metadata"]
            }
            for d in compressed
        ]
    }